In [4]:
import os
import sys

# This finds the absolute path of the folder ONE level above your notebook
# which should be your project root containing the 'utils' folder.
root_path = os.path.abspath(os.path.join('..'))

if root_path not in sys.path:
    sys.path.append(root_path)

import pandas as pd

# 1. Load the two datasets
df_diffstat = pd.read_csv('../data/intermediate/churn_tosem.csv')
df_semantic = pd.read_csv('../data/intermediate/churn_tosem_semantic.csv')

# 2. Merge on commit_id
# We want to see how these two methods differ for the exact same commits
df_combined = pd.merge(
    df_diffstat, 
    df_semantic, 
    on='commit_id', 
    suffixes=('_diffstat', '_semantic')
)
# 3. Sample commits where our semantic tool found modifications
# This is where we expect diffstat to be "blind"
sample_df = df_combined[df_combined['lines_modified_semantic'] > 0].sample(30, random_state=42)

In [5]:
from utils.git import get_context_all_metrics

verification_results = []

for _, row in sample_df.iterrows():
    c_id = row['commit_id']
    repo_name = row['project_diffstat']
    
    # Get Ground Truth from your utils/git.py method
    gt_mod, gt_add, gt_rem = get_context_all_metrics(c_id, repo_name)
    
    if gt_mod is not None:
        verification_results.append({
            "Commit": c_id[:8],
            "Project": repo_name,
            "Diffstat_Mod": row['lines_modified_diffstat'],
            "Semantic_Mod": row['lines_modified_semantic'],
            "GT_Mod": gt_mod, # From git show -c
            "Diffstat_Add": row['lines_added_diffstat'],
            "Semantic_Add": row['lines_added_semantic'],
            "GT_Add": gt_add,
            "Diffstat_Rem": row['lines_removed_diffstat'],
            "Semantic_Rem": row['lines_removed_semantic'],
            "GT_Rem": gt_rem
        })

df_verify = pd.DataFrame(verification_results)

In [6]:
# 1. Add Error Calculation Columns (to prove your tool works)
# Error = |Tool_Result - Ground_Truth|
df_verify['Error_Diffstat'] = (df_verify['Diffstat_Mod'] - df_verify['GT_Mod']).abs()
df_verify['Error_Semantic'] = (df_verify['Semantic_Mod'] - df_verify['GT_Mod']).abs()

# 2. Display the full table
# This will show Additions, Removals, and Modifications for all three methods
pd.set_option('display.max_columns', None)  # Ensure we don't hide columns
display(df_verify)

# 3. Quick Summary Stats (The "Money" Numbers for the paper)
print("\n--- PERFORMANCE SUMMARY ---")
print(f"Average Diffstat Error (Modifications): {df_verify['Error_Diffstat'].mean():.2f} lines")
print(f"Average Semantic Tool Error (Modifications): {df_verify['Error_Semantic'].mean():.2f} lines")
print(f"Correlation (Semantic vs Ground Truth): {df_verify['Semantic_Mod'].corr(df_verify['GT_Mod']):.4f}")

,Commit,Project,Diffstat_Mod,Semantic_Mod,GT_Mod,Diffstat_Add,Semantic_Add,GT_Add,Diffstat_Rem,Semantic_Rem,GT_Rem,Error_Diffstat,Error_Semantic
0,29f975ea,commons-compress,28,24,33,9,13,2,54,58,50,5,9
1,ed3823b0,spring-framework,53,48,71,599,604,1141,18,23,25,18,23
2,49a3f0a1,lucene-solr,9,8,39,15,16,48,3,4,13,30,31
3,31815598,netty,23,23,31,0,0,0,0,0,0,8,8
4,38a88233,JGroups,199,170,401,1151,1180,2147,1631,1660,2058,202,231
5,899b8b0d,spring-data-jpa,16,7,8,86,95,192,10,19,16,8,1
6,784fabac,okhttp,36,12,17,158,182,443,32,56,1,19,5
7,54b65c1d,camel,4,3,7,3,4,6,7,8,3,3,4
8,2a2f1dc4,commons-compress,1,1,1,0,0,0,0,0,0,0,0
9,5e1d70c6,camel,1,1,3,25,25,166,0,0,0,2,2



--- PERFORMANCE SUMMARY ---
Average Diffstat Error (Modifications): 24.70 lines
Average Semantic Tool Error (Modifications): 27.97 lines
Correlation (Semantic vs Ground Truth): 0.9315
